In [10]:
# =========================
# CS-340 Grazioso Dashboard
# =========================

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# OS / data / plotting
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

JupyterDash.infer_jupyter_proxy_config()

# -------------------------
# CRUD / Database Connection
# -------------------------
from animal_shelter import AnimalShelter

username = "aacuser"
password = "Coderzretreet"

db = AnimalShelter(username, password)

# Initial full dataset
df = pd.DataFrame.from_records(db.read({}))
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# -------------------------
# App Setup
# -------------------------
app = JupyterDash(__name__)

# Logo (replace with your own file if needed)
image_filename = 'Grazioso Salvare Logo.png'  # ensure this file exists in your workspace
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Header
    html.Center(html.B(html.H1('CS-340 Dashboard - Russell Buck'))),
    html.Center(html.Img(
        src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={'height': '80px'}
    )),
    html.Center(html.A("Grazioso Salvare (SNHU)", href="https://www.snhu.edu", target="_blank")),
    html.Hr(),

    # Filter controls
    html.Div([
        html.H4("Rescue Type Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True
        )
    ], style={'margin': '10px'}),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        }
    ),

    html.Br(),
    html.Hr(),

    # Charts row
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'flex': '1', 'padding': '10px'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'flex': '1', 'padding': '10px'}
            )
        ]
    )
])

# -------------------------
# Callbacks
# -------------------------

# Filter DataTable based on rescue type
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'water':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'mountain':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'disaster':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:  # reset
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Chart (pie chart of breed distribution)
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None:
        return [html.H4("No data available")]

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty or 'breed' not in dff.columns:
        return [html.H4("No data available for this filter")]

    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution for Selected Rescue Type'
    )

    return [dcc.Graph(figure=fig)]


# Highlight selected column in DataTable
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]


# Map update based on selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or index is None or len(index) == 0:
        return [html.H4("Select a row to view location")]

    dff = pd.DataFrame.from_dict(viewData)

    # Selected row index
    row = index[0]

    # Extract coordinates from your dataset
    lat = dff.iloc[row, 13]
    lon = dff.iloc[row, 14]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],      # <-- CENTER ON THE PIN
            zoom=14,                # <-- Slight zoom-in for clarity
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# -------------------------
# Run App
# -------------------------
app.run_server(mode='external', debug=False)


Connected to MongoDB successfully.


 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [19/Jun/2026 00:56:00] "GET /_alive_bef83549-4a80-4acd-93f8-f7d02050fd2b HTTP/1.1" 200 -


Dash app running on https://yesobserve-robinobscure-3000.codio.io/proxy/8050/


127.0.0.1 - - [19/Jun/2026 00:56:06] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:06] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:06] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:07] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Jun/2026 00:56:07] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:07] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:07] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:07] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Jun/2026 00:56:07] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:08] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:08] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Jun/2026 00:56:08] "GET /_dash-component-sui